<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Evaluation_Benchmarking_Integration_CNN_DailyMail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation, Benchmarking, and Integration

## Evaluation of LLMs for Text Summarization

This notebook evaluates three language models on the CNN/DailyMail summarization dataset:

- `t5-small`;
- `t5-base`;
- `gpt2`.

It includes:

- dataset loading and exploration;
- batched summary generation;
- strict and customized accuracy metrics;
- ROUGE evaluation;
- ROUGE behavior experiments;
- per-row model evaluation;
- model comparison tables and visualizations;
- analytical conclusions.

The Hugging Face dataset columns are adapted as follows:

| Original column | Course column |
|---|---|
| `article` | `prompt_text` |
| `highlights` | `prompt_title` |

## Part I — Setup

In [ ]:
%pip install -q \
    "rouge_score==0.1.2" \
    evaluate \
    accelerate \
    datasets \
    nltk \
    transformers \
    sentencepiece \
    pandas \
    matplotlib

In [ ]:
import gc
import re
import string
import time
import warnings
from typing import Iterator, Sequence

import evaluate
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from IPython.display import display
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    T5ForConditionalGeneration,
)

warnings.filterwarnings("ignore")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Execution device:", DEVICE)

## Part II — Dataset Loading and Exploration

The notebook uses `abisee/cnn_dailymail`, configuration `3.0.0`.

To reduce the computational load, only:

- 100 training examples;
- 50 test examples;

are loaded.

In [ ]:
DATASET_NAME = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"
TRAIN_SAMPLE_SIZE = 100
TEST_SAMPLE_SIZE = 50

train_dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=f"train[:{TRAIN_SAMPLE_SIZE}]",
)

test_dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=f"test[:{TEST_SAMPLE_SIZE}]",
)

print(train_dataset)
print(test_dataset)
print("\nOriginal columns:", train_dataset.column_names)

In [ ]:
train_sample = train_dataset.to_pandas().rename(
    columns={
        "article": "prompt_text",
        "highlights": "prompt_title",
    }
)

test_sample = test_dataset.to_pandas().rename(
    columns={
        "article": "prompt_text",
        "highlights": "prompt_title",
    }
)

required_columns = {
    "prompt_text",
    "prompt_title",
}

if not required_columns.issubset(train_sample.columns):
    raise ValueError(
        "The training sample does not contain the required columns."
    )

if not required_columns.issubset(test_sample.columns):
    raise ValueError(
        "The test sample does not contain the required columns."
    )

train_sample = (
    train_sample
    .dropna(subset=["prompt_text", "prompt_title"])
    .reset_index(drop=True)
)

test_sample = (
    test_sample
    .dropna(subset=["prompt_text", "prompt_title"])
    .reset_index(drop=True)
)

print("Training sample shape:", train_sample.shape)
print("Test sample shape:", test_sample.shape)

In [ ]:
print("FIRST TRAINING EXAMPLE")
print("=" * 100)

print("\nARTICLE")
print(train_sample.loc[0, "prompt_text"])

print("\nREFERENCE SUMMARY")
print(train_sample.loc[0, "prompt_title"])

In [ ]:
print("TRAINING SAMPLE")
display(train_sample.head())

print("\nTEST SAMPLE")
display(test_sample.head())

## Part III — Summarization with T5

T5 receives each article with the prefix `summarize:`.

The implementation includes:

- GPU support;
- batch processing;
- input truncation;
- beam search;
- memory cleanup after every batch;
- progress information.

In [ ]:
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def batch_generator(
    items: Sequence[str],
    batch_size: int,
) -> Iterator[list[str]]:
    if batch_size <= 0:
        raise ValueError(
            "batch_size must be greater than zero."
        )

    for start in range(0, len(items), batch_size):
        yield list(items[start:start + batch_size])


def clear_memory() -> None:
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def summarize_with_t5(
    texts: Sequence[str],
    model_name: str = "t5-small",
    batch_size: int = 8,
    max_input_length: int = 512,
    max_new_tokens: int = 64,
    num_beams: int = 4,
) -> list[str]:
    texts = [
        clean_text(text)
        for text in texts
    ]

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = T5ForConditionalGeneration.from_pretrained(
        model_name
    )

    model.to(DEVICE)
    model.eval()

    summaries = []
    total_batches = int(
        np.ceil(len(texts) / batch_size)
    )

    start_time = time.time()

    try:
        for batch_index, text_batch in enumerate(
            batch_generator(texts, batch_size),
            start=1,
        ):
            model_inputs = [
                f"summarize: {text}"
                for text in text_batch
            ]

            encoded_inputs = tokenizer(
                model_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_input_length,
            )

            encoded_inputs = {
                key: value.to(DEVICE)
                for key, value in encoded_inputs.items()
            }

            with torch.no_grad():
                generated_ids = model.generate(
                    **encoded_inputs,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    no_repeat_ngram_size=3,
                    early_stopping=True,
                )

            batch_summaries = tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            summaries.extend(
                clean_text(summary)
                for summary in batch_summaries
            )

            print(
                f"{model_name}: "
                f"batch {batch_index}/{total_batches}"
            )

            del encoded_inputs
            del generated_ids
            clear_memory()

    finally:
        elapsed_time = time.time() - start_time
        print(
            f"{model_name} completed in "
            f"{elapsed_time:.2f} seconds."
        )

        model.to("cpu")
        del model
        del tokenizer
        clear_memory()

    return summaries

In [ ]:
T5_SMALL_BATCH_SIZE = (
    8 if torch.cuda.is_available() else 2
)

train_sample["t5_small_summary"] = (
    summarize_with_t5(
        texts=train_sample["prompt_text"].tolist(),
        model_name="t5-small",
        batch_size=T5_SMALL_BATCH_SIZE,
        max_input_length=512,
        max_new_tokens=64,
        num_beams=4,
    )
)

t5_small_display = train_sample[
    [
        "prompt_title",
        "t5_small_summary",
    ]
].copy()

t5_small_display.columns = [
    "reference_summary",
    "t5_small_generated_summary",
]

display(t5_small_display.head(10))

## Part IV — Accuracy Evaluation

Exact-match accuracy is very strict. A generated summary is counted as correct only when it is identical to the reference.

Two additional metrics are included:

- normalized exact-match accuracy;
- custom token-overlap accuracy based on Jaccard similarity.

In [ ]:
def exact_match_accuracy(
    predictions: Sequence[str],
    references: Sequence[str],
) -> float:
    if len(predictions) != len(references):
        raise ValueError(
            "Predictions and references must have "
            "the same length."
        )

    if len(predictions) == 0:
        return 0.0

    matches = [
        str(prediction) == str(reference)
        for prediction, reference in zip(
            predictions,
            references,
        )
    ]

    return float(np.mean(matches))


def normalize_text(text: str) -> str:
    text = clean_text(text).lower()

    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation,
        )
    )

    return clean_text(text)


def normalized_exact_match_accuracy(
    predictions: Sequence[str],
    references: Sequence[str],
) -> float:
    normalized_predictions = [
        normalize_text(text)
        for text in predictions
    ]

    normalized_references = [
        normalize_text(text)
        for text in references
    ]

    return exact_match_accuracy(
        normalized_predictions,
        normalized_references,
    )


def token_jaccard_similarity(
    prediction: str,
    reference: str,
) -> float:
    prediction_tokens = set(
        normalize_text(prediction).split()
    )

    reference_tokens = set(
        normalize_text(reference).split()
    )

    union = prediction_tokens.union(
        reference_tokens
    )

    if not union:
        return 1.0

    intersection = prediction_tokens.intersection(
        reference_tokens
    )

    return len(intersection) / len(union)


def custom_summary_accuracy(
    predictions: Sequence[str],
    references: Sequence[str],
    similarity_threshold: float = 0.30,
) -> float:
    if len(predictions) != len(references):
        raise ValueError(
            "Predictions and references must have "
            "the same length."
        )

    if len(predictions) == 0:
        return 0.0

    decisions = [
        token_jaccard_similarity(
            prediction,
            reference,
        ) >= similarity_threshold
        for prediction, reference in zip(
            predictions,
            references,
        )
    ]

    return float(np.mean(decisions))

In [ ]:
reference_summaries = (
    train_sample["prompt_title"].tolist()
)

t5_small_predictions = (
    train_sample["t5_small_summary"].tolist()
)

strict_accuracy = exact_match_accuracy(
    t5_small_predictions,
    reference_summaries,
)

normalized_accuracy = (
    normalized_exact_match_accuracy(
        t5_small_predictions,
        reference_summaries,
    )
)

custom_accuracy = custom_summary_accuracy(
    t5_small_predictions,
    reference_summaries,
    similarity_threshold=0.30,
)

print(
    f"Strict exact-match accuracy: "
    f"{strict_accuracy:.4f}"
)

print(
    f"Normalized exact-match accuracy: "
    f"{normalized_accuracy:.4f}"
)

print(
    f"Custom token-overlap accuracy: "
    f"{custom_accuracy:.4f}"
)

### Accuracy interpretation

Exact-match accuracy is usually very low or zero because summarization allows many valid formulations.

For example:

- Reference: `The company reports strong quarterly growth.`
- Prediction: `Strong growth was reported by the company this quarter.`

The meaning is similar, but strict accuracy gives a score of zero. ROUGE is therefore more suitable for measuring partial lexical overlap.

## Part V — ROUGE Metric Implementation

ROUGE compares generated summaries with reference summaries.

Sentence tokenization and newline formatting are used because `rougeLsum` evaluates summary-level sentence structure.

In [ ]:
rouge_metric = evaluate.load("rouge")


def format_for_rouge(text: str) -> str:
    text = clean_text(text)

    if not text:
        return ""

    sentences = sent_tokenize(text)
    return "\n".join(sentences)


def compute_rouge_score(
    predictions: Sequence[str],
    references: Sequence[str],
    use_stemmer: bool = True,
) -> dict[str, float]:
    if len(predictions) != len(references):
        raise ValueError(
            "Predictions and references must have "
            "the same length."
        )

    formatted_predictions = [
        format_for_rouge(text)
        for text in predictions
    ]

    formatted_references = [
        format_for_rouge(text)
        for text in references
    ]

    scores = rouge_metric.compute(
        predictions=formatted_predictions,
        references=formatted_references,
        use_stemmer=use_stemmer,
    )

    return {
        metric: float(value)
        for metric, value in scores.items()
    }


t5_small_rouge = compute_rouge_score(
    predictions=t5_small_predictions,
    references=reference_summaries,
    use_stemmer=True,
)

print("T5-small aggregate ROUGE scores")

for metric, value in t5_small_rouge.items():
    print(f"{metric}: {value:.4f}")

## Part VI — Understanding ROUGE Scores

In [ ]:
exact_match_test = compute_rouge_score(
    predictions=reference_summaries,
    references=reference_summaries,
)

empty_prediction_test = compute_rouge_score(
    predictions=[""] * len(reference_summaries),
    references=reference_summaries,
)

boundary_test_results = pd.DataFrame(
    [
        {
            "test": "Exact match",
            **exact_match_test,
        },
        {
            "test": "Empty predictions",
            **empty_prediction_test,
        },
    ]
)

display(boundary_test_results)

In [ ]:
stemming_prediction = [
    "The runner runs and connects systems."
]

stemming_reference = [
    "A running athlete connected the systems."
]

stemming_results = pd.DataFrame(
    [
        {
            "configuration": "Without stemming",
            **compute_rouge_score(
                stemming_prediction,
                stemming_reference,
                use_stemmer=False,
            ),
        },
        {
            "configuration": "With stemming",
            **compute_rouge_score(
                stemming_prediction,
                stemming_reference,
                use_stemmer=True,
            ),
        },
    ]
)

display(stemming_results)

In [ ]:
overlap_reference = [
    "the cat sat on the mat"
]

overlap_examples = {
    "Exact overlap":
        "the cat sat on the mat",
    "High unigram overlap":
        "the cat rested on a mat",
    "Same words, different order":
        "mat the on sat cat the",
    "Low overlap":
        "a dog played outside",
    "No overlap":
        "computers process digital information",
}

overlap_rows = []

for label, prediction in overlap_examples.items():
    scores = compute_rouge_score(
        predictions=[prediction],
        references=overlap_reference,
        use_stemmer=False,
    )

    overlap_rows.append(
        {
            "example": label,
            "prediction": prediction,
            **scores,
        }
    )

ngram_overlap_results = pd.DataFrame(
    overlap_rows
)

display(ngram_overlap_results)

In [ ]:
symmetry_prediction = [
    "the cat sat on the mat"
]

symmetry_reference = [
    "the cat was sitting on a mat"
]

symmetry_results = pd.DataFrame(
    [
        {
            "direction": "prediction → reference",
            **compute_rouge_score(
                symmetry_prediction,
                symmetry_reference,
            ),
        },
        {
            "direction": "reference → prediction",
            **compute_rouge_score(
                symmetry_reference,
                symmetry_prediction,
            ),
        },
    ]
)

display(symmetry_results)

### ROUGE observations

- Exact matches produce scores of 1.
- Empty predictions produce scores of 0.
- Stemming may improve scores when words share the same root.
- ROUGE-1 measures unigram overlap.
- ROUGE-2 measures bigram overlap and is stricter.
- ROUGE-L uses the longest common subsequence.
- The F1-style ROUGE scores are usually symmetric when predictions and references are exchanged, even though precision and recall interpretations are reversed.

## Part VII — Comparing T5-small, T5-base, and GPT-2

`t5-base` is larger than `t5-small` and usually requires more memory.

GPT-2 is a decoder-only model, not a model specifically trained for summarization. It receives a `TL;DR:` prompt, and only newly generated tokens are retained.

In [ ]:
T5_BASE_BATCH_SIZE = (
    4 if torch.cuda.is_available() else 1
)

train_sample["t5_base_summary"] = (
    summarize_with_t5(
        texts=train_sample["prompt_text"].tolist(),
        model_name="t5-base",
        batch_size=T5_BASE_BATCH_SIZE,
        max_input_length=512,
        max_new_tokens=64,
        num_beams=4,
    )
)

display(
    train_sample[
        [
            "prompt_title",
            "t5_small_summary",
            "t5_base_summary",
        ]
    ].head(10)
)

In [ ]:
def summarize_with_gpt2(
    texts: Sequence[str],
    model_name: str = "gpt2",
    batch_size: int = 4,
    max_input_length: int = 768,
    max_new_tokens: int = 64,
    num_beams: int = 2,
) -> list[str]:
    texts = [
        clean_text(text)
        for text in texts
    ]

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name
    )

    model.config.pad_token_id = (
        tokenizer.pad_token_id
    )

    model.to(DEVICE)
    model.eval()

    summaries = []
    total_batches = int(
        np.ceil(len(texts) / batch_size)
    )

    start_time = time.time()

    try:
        for batch_index, text_batch in enumerate(
            batch_generator(texts, batch_size),
            start=1,
        ):
            prompts = [
                (
                    "Summarize the following article "
                    "in one concise sentence.\n\n"
                    f"Article: {text}\n\n"
                    "TL;DR:"
                )
                for text in text_batch
            ]

            encoded_inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_input_length,
            )

            encoded_inputs = {
                key: value.to(DEVICE)
                for key, value in encoded_inputs.items()
            }

            prompt_width = (
                encoded_inputs["input_ids"].shape[1]
            )

            with torch.no_grad():
                generated_ids = model.generate(
                    **encoded_inputs,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    no_repeat_ngram_size=3,
                    early_stopping=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            continuation_ids = (
                generated_ids[:, prompt_width:]
            )

            batch_summaries = tokenizer.batch_decode(
                continuation_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            summaries.extend(
                clean_text(summary)
                for summary in batch_summaries
            )

            print(
                f"{model_name}: "
                f"batch {batch_index}/{total_batches}"
            )

            del encoded_inputs
            del generated_ids
            del continuation_ids
            clear_memory()

    finally:
        elapsed_time = time.time() - start_time
        print(
            f"{model_name} completed in "
            f"{elapsed_time:.2f} seconds."
        )

        model.to("cpu")
        del model
        del tokenizer
        clear_memory()

    return summaries

In [ ]:
GPT2_BATCH_SIZE = (
    4 if torch.cuda.is_available() else 1
)

train_sample["gpt2_summary"] = (
    summarize_with_gpt2(
        texts=train_sample["prompt_text"].tolist(),
        model_name="gpt2",
        batch_size=GPT2_BATCH_SIZE,
        max_input_length=768,
        max_new_tokens=64,
        num_beams=2,
    )
)

display(
    train_sample[
        [
            "prompt_title",
            "t5_small_summary",
            "t5_base_summary",
            "gpt2_summary",
        ]
    ].head(10)
)

### Per-row ROUGE

The aggregate `evaluate` metric is used for overall scores. The underlying `rouge_score` package is used for efficient row-by-row scoring.

In [ ]:
def compute_rouge_per_row(
    dataframe: pd.DataFrame,
    prediction_column: str,
    reference_column: str = "prompt_title",
    prefix: str = "model",
    use_stemmer: bool = True,
) -> pd.DataFrame:
    scorer = rouge_scorer.RougeScorer(
        [
            "rouge1",
            "rouge2",
            "rougeL",
            "rougeLsum",
        ],
        use_stemmer=use_stemmer,
    )

    score_rows = []

    for prediction, reference in zip(
        dataframe[prediction_column],
        dataframe[reference_column],
    ):
        prediction = format_for_rouge(
            prediction
        )

        reference = format_for_rouge(
            reference
        )

        scores = scorer.score(
            reference,
            prediction,
        )

        score_rows.append(
            {
                f"{prefix}_rouge1":
                    scores["rouge1"].fmeasure,
                f"{prefix}_rouge2":
                    scores["rouge2"].fmeasure,
                f"{prefix}_rougeL":
                    scores["rougeL"].fmeasure,
                f"{prefix}_rougeLsum":
                    scores["rougeLsum"].fmeasure,
            }
        )

    return pd.DataFrame(score_rows)

In [ ]:
t5_small_per_row = compute_rouge_per_row(
    dataframe=train_sample,
    prediction_column="t5_small_summary",
    prefix="t5_small",
)

t5_base_per_row = compute_rouge_per_row(
    dataframe=train_sample,
    prediction_column="t5_base_summary",
    prefix="t5_base",
)

gpt2_per_row = compute_rouge_per_row(
    dataframe=train_sample,
    prediction_column="gpt2_summary",
    prefix="gpt2",
)

per_row_results = pd.concat(
    [
        train_sample[
            [
                "id",
                "prompt_title",
                "t5_small_summary",
                "t5_base_summary",
                "gpt2_summary",
            ]
        ].reset_index(drop=True),
        t5_small_per_row,
        t5_base_per_row,
        gpt2_per_row,
    ],
    axis=1,
)

display(per_row_results.head(10))

## Part VIII — Comparing All Models

In [ ]:
MODEL_COLUMNS = {
    "t5-small": "t5_small_summary",
    "t5-base": "t5_base_summary",
    "gpt2": "gpt2_summary",
}


def compare_models(
    dataframe: pd.DataFrame,
    model_columns: dict[str, str],
    reference_column: str = "prompt_title",
) -> pd.DataFrame:
    comparison_rows = []

    references = (
        dataframe[reference_column].tolist()
    )

    for model_name, prediction_column in (
        model_columns.items()
    ):
        predictions = (
            dataframe[prediction_column].tolist()
        )

        rouge_scores = compute_rouge_score(
            predictions=predictions,
            references=references,
            use_stemmer=True,
        )

        comparison_rows.append(
            {
                "model": model_name,
                "strict_accuracy":
                    exact_match_accuracy(
                        predictions,
                        references,
                    ),
                "normalized_accuracy":
                    normalized_exact_match_accuracy(
                        predictions,
                        references,
                    ),
                "custom_accuracy":
                    custom_summary_accuracy(
                        predictions,
                        references,
                        similarity_threshold=0.30,
                    ),
                **rouge_scores,
            }
        )

    results = pd.DataFrame(
        comparison_rows
    )

    return results.sort_values(
        by="rougeLsum",
        ascending=False,
    ).reset_index(drop=True)

In [ ]:
def compare_models_summaries(
    dataframe: pd.DataFrame,
    number_of_examples: int = 10,
) -> pd.DataFrame:
    number_of_examples = min(
        number_of_examples,
        len(dataframe),
    )

    comparison = dataframe.loc[
        :number_of_examples - 1,
        [
            "prompt_title",
            "t5_small_summary",
            "t5_base_summary",
            "gpt2_summary",
        ],
    ].copy()

    comparison.columns = [
        "reference_summary",
        "t5_small",
        "t5_base",
        "gpt2",
    ]

    return comparison

In [ ]:
model_comparison = compare_models(
    dataframe=train_sample,
    model_columns=MODEL_COLUMNS,
)

summary_comparison = compare_models_summaries(
    dataframe=train_sample,
    number_of_examples=10,
)

print("AGGREGATED MODEL SCORES")
display(model_comparison)

print("\nSIDE-BY-SIDE SUMMARIES")
display(summary_comparison)

In [ ]:
rouge_columns = [
    "rouge1",
    "rouge2",
    "rougeL",
    "rougeLsum",
]

chart_data = (
    model_comparison
    .set_index("model")[rouge_columns]
)

ax = chart_data.plot(
    kind="bar",
    figsize=(11, 6),
)

ax.set_title(
    "Average ROUGE Scores by Model"
)

ax.set_xlabel("Model")
ax.set_ylabel("ROUGE F1 score")
ax.set_ylim(0, 1)
ax.tick_params(
    axis="x",
    rotation=0,
)

plt.tight_layout()
plt.show()

In [ ]:
best_model_row = model_comparison.iloc[0]

print(
    "Best model according to ROUGE-Lsum:",
    best_model_row["model"],
)

print(
    "Best ROUGE-Lsum score:",
    round(
        float(best_model_row["rougeLsum"]),
        4,
    ),
)

print("\nGeneral interpretation:")

print(
    "- Exact-match accuracy is expected to be "
    "very low for every model."
)

print(
    "- ROUGE provides a more informative "
    "comparison because it rewards partial overlap."
)

print(
    "- T5 models are designed for text-to-text "
    "tasks and should generally summarize more "
    "reliably than GPT-2."
)

print(
    "- T5-base may improve quality compared with "
    "T5-small, but it requires more memory and "
    "computation."
)

print(
    "- GPT-2 may repeat the prompt, produce "
    "incomplete output, or continue the article "
    "instead of creating a focused summary."
)

## Final Analytical Report

### Accuracy

Strict accuracy is unsuitable as the main metric for summarization because valid summaries rarely match a reference word for word. Normalization removes superficial punctuation and capitalization differences, while the custom overlap metric gives partial credit when enough important words are shared.

### ROUGE

ROUGE is more useful than exact accuracy for this task:

- ROUGE-1 evaluates individual word overlap;
- ROUGE-2 evaluates two-word sequence overlap;
- ROUGE-L evaluates the longest common subsequence;
- ROUGE-Lsum evaluates summary-level sentence structure.

ROUGE still has limitations. It focuses on lexical overlap and may underrate a correct paraphrase that uses different vocabulary. It can also reward a summary that copies many words without preserving the correct meaning.

### Model comparison

T5 is trained with a text-to-text objective and can follow the explicit `summarize:` instruction. GPT-2 is primarily trained to continue text, so the `TL;DR:` prompt is only an indirect way to request a summary.

The comparison should therefore consider:

- ROUGE performance;
- factual correctness;
- conciseness;
- coherence;
- computational cost;
- generation speed.

### Model size

A larger model is not automatically the best choice. `t5-base` may produce stronger summaries than `t5-small`, but it uses more memory and runs more slowly. In a constrained environment, `t5-small` may offer the best balance between quality and efficiency.

### Recommended evaluation strategy

A reliable summarization evaluation should combine:

1. automatic metrics such as ROUGE;
2. per-row error inspection;
3. side-by-side comparison;
4. human checks for factual accuracy and readability;
5. runtime and resource measurements.

## Export the Results

The following files can be downloaded from the Colab file browser after execution.

In [ ]:
model_comparison.to_csv(
    "model_comparison_scores.csv",
    index=False,
)

per_row_results.to_csv(
    "per_row_rouge_results.csv",
    index=False,
)

summary_comparison.to_csv(
    "side_by_side_summaries.csv",
    index=False,
)

print("Exported files:")
print("- model_comparison_scores.csv")
print("- per_row_rouge_results.csv")
print("- side_by_side_summaries.csv")